# QLoRA Fine-Tuning for RAG Chatbot

This notebook walks through fine-tuning **Mistral 7B** using QLoRA (Quantized Low-Rank Adaptation) on Google Colab, then converting the result to GGUF format for use with Ollama in your local RAG chatbot.

## What this notebook does
1. Loads Mistral 7B in 4-bit quantization to fit in Colab's free GPU
2. Attaches lightweight LoRA adapter layers (only ~0.5% of parameters are trained)
3. Fine-tunes on your custom JSONL training data
4. Converts the merged model to GGUF format (Q4_K_M quantization)
5. Provides instructions to import the model into Ollama locally

## Hardware requirements
- **Minimum:** T4 GPU (16 GB VRAM) — available on Colab free tier
- **Recommended:** L4 (24 GB) or A100 (40 GB) via Colab Pro for faster training
- Go to **Runtime → Change runtime type → T4 GPU** before running

## Estimated time (T4 GPU)
| Dataset size | Epochs | Estimated time |
|---|---|---|
| ~500 examples | 3 | ~20–30 min |
| ~2 000 examples | 3 | ~1–2 hours |
| ~10 000 examples | 3 | ~5–8 hours |

> Run each cell in order with **Shift+Enter**.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    print(f"GPU : {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb >= 15:
        print("Status: OK — enough VRAM for Mistral 7B with QLoRA")
    else:
        print("Warning: Less than 15 GB VRAM detected. Reduce LORA_R or MAX_SEQ_LEN if you hit OOM errors.")
else:
    print("ERROR: No GPU detected!")
    print("Go to Runtime -> Change runtime type -> T4 GPU and re-run.")

In [ ]:
!pip install -q peft transformers bitsandbytes accelerate datasets trl

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
BASE_MODEL    = "mistralai/Mistral-7B-v0.1"
OUTPUT_DIR    = "/content/finetuned-model"

# ── LoRA hyper-parameters ──────────────────────────────────────────────────
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

# ── Training hyper-parameters ──────────────────────────────────────────────
MAX_SEQ_LEN   = 2048
BATCH_SIZE    = 4
GRAD_ACCUM    = 4          # effective batch = BATCH_SIZE * GRAD_ACCUM
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3

# ── Hugging Face token (required for gated models like Llama) ──────────────
HF_TOKEN      = ""         # required for gated models like Llama

print("Configuration:")
print(f"  BASE_MODEL    = {BASE_MODEL}")
print(f"  OUTPUT_DIR    = {OUTPUT_DIR}")
print(f"  LORA_R        = {LORA_R}")
print(f"  LORA_ALPHA    = {LORA_ALPHA}")
print(f"  LORA_DROPOUT  = {LORA_DROPOUT}")
print(f"  MAX_SEQ_LEN   = {MAX_SEQ_LEN}")
print(f"  BATCH_SIZE    = {BATCH_SIZE}")
print(f"  GRAD_ACCUM    = {GRAD_ACCUM}")
print(f"  LEARNING_RATE = {LEARNING_RATE}")
print(f"  NUM_EPOCHS    = {NUM_EPOCHS}")

## Step 1: Upload Training Data

Upload a **JSONL** file where each line is a JSON object with the keys `instruction`, `context`, and `response`.

Example line:
```json
{"instruction": "What is the refund policy?", "context": "Customers may return items within 30 days.", "response": "You can return items within 30 days of purchase for a full refund."}
```

See `docs/TRAINING_DATA_GUIDE.md` for tips on building a high-quality dataset.

In [ ]:
import json
from google.colab import files

print("Select your training JSONL file to upload...")
uploaded = files.upload()

# Save the uploaded file to a fixed path
upload_name = list(uploaded.keys())[0]
with open("/content/training.jsonl", "wb") as f:
    f.write(uploaded[upload_name])
print(f"Saved as /content/training.jsonl ({len(uploaded[upload_name])} bytes)")

# Preview the first 3 examples
print("\nFirst 3 examples:")
with open("/content/training.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        example = json.loads(line.strip())
        print(f"\n--- Example {i + 1} ---")
        for key, value in example.items():
            print(f"  {key}: {str(value)[:120]}")

## Step 2: Load Model with 4-bit Quantization

We use **BitsAndBytes NF4** quantization to compress Mistral 7B from ~14 GB down to ~4 GB, making it fit comfortably on a T4 GPU.

Key settings:
- `load_in_4bit=True` — store weights in 4-bit
- `bnb_4bit_use_double_quant=True` — quantize the quantization constants too (saves ~0.4 bits/param)
- `bnb_4bit_quant_type="nf4"` — NormalFloat4, optimal for normally-distributed weights
- `bnb_4bit_compute_dtype=torch.bfloat16` — use bfloat16 for matrix multiplications

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load tokenizer
print(f"Loading tokenizer from {BASE_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN or None,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model in 4-bit
print(f"Loading model in 4-bit quantization ...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN or None,
    trust_remote_code=True,
)

# Prepare for k-bit training (freezes base weights, enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

vram_used = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"Model loaded. GPU memory in use: {vram_used:.1f} GB")

## Step 3: Configure LoRA Adapters

LoRA injects small trainable rank-decomposition matrices into the attention and MLP layers. Only these adapter weights are updated during training — the base model stays frozen.

- **`r` (rank):** Controls adapter capacity. Higher = more expressive but more VRAM.
- **`lora_alpha`:** Scaling factor. Typically set to `2 × r`.
- **`target_modules`:** Which weight matrices to adapt. We target all projection layers.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

# Report trainable parameter count
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params       = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters : {trainable_params:,}")
print(f"All parameters       : {all_params:,}")
print(f"Trainable %          : {100 * trainable_params / all_params:.4f}%")

## Step 4: Prepare Dataset

Each JSONL example is formatted using the Alpaca-style prompt template:

```
### Instruction:
{instruction}

### Context:
{context}

### Response:
{response}
```

The dataset is then tokenized and split 90 % train / 10 % eval.

In [ ]:
from datasets import load_dataset

# Load the uploaded JSONL file
raw_dataset = load_dataset("json", data_files="/content/training.jsonl", split="train")
print(f"Loaded {len(raw_dataset)} examples")

def format_example(example):
    """Format a single example into the Alpaca-style prompt."""
    text = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Context:\n{example.get('context', '')}\n\n"
        f"### Response:\n{example['response']}"
    )
    return {"text": text}

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )

# Format and tokenize
formatted = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)
tokenized = formatted.map(tokenize, batched=True)

# 90 / 10 train-eval split
split = tokenized.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train examples : {len(train_dataset)}")
print(f"Eval examples  : {len(eval_dataset)}")
print(f"\nSample formatted text (first 300 chars):")
print(formatted[0]["text"][:300])

## Step 5: Train

We use Hugging Face `Trainer` with the configuration defined earlier. Watch the `loss` column — it should decrease steadily. A final loss below 1.0 generally indicates good learning.

The trained adapter and tokenizer are saved to `OUTPUT_DIR` when training completes.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

# Save the fine-tuned adapter and tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel and tokenizer saved to {OUTPUT_DIR}")

## Step 6: Convert to GGUF and Download

We merge the LoRA adapters back into the base model weights, then use **llama.cpp** to convert and quantize to GGUF format (Q4_K_M — a good balance of size and quality).

The resulting `.gguf` file can be imported directly into Ollama.

In [ ]:
import gc
import os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM
from google.colab import files

MERGED_DIR = "/content/merged-model"
GGUF_PATH  = "/content/model.gguf"

# ── 1. Free training memory ────────────────────────────────────────────────
del trainer
gc.collect()
torch.cuda.empty_cache()

# ── 2. Reload base model in fp16 and merge adapters ────────────────────────
print("Reloading base model in fp16 for merging (this takes a few minutes)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN or None,
    trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

# ── 3. Install llama.cpp ───────────────────────────────────────────────────
print("\nCloning llama.cpp...")
!git clone -q https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

# ── 4. Convert to GGUF ────────────────────────────────────────────────────
print("\nConverting to GGUF (Q4_K_M)...")
!python /content/llama.cpp/convert_hf_to_gguf.py \
    {MERGED_DIR} \
    --outfile {GGUF_PATH} \
    --outtype q4_K_M

gguf_size_gb = os.path.getsize(GGUF_PATH) / (1024 ** 3)
print(f"\nGGUF file created: {GGUF_PATH}")
print(f"File size: {gguf_size_gb:.2f} GB")

# ── 5. Download to local machine ──────────────────────────────────────────
print("\nStarting download...")
files.download(GGUF_PATH)

## Step 7: Import into Ollama (run on your local machine)

After downloading `model.gguf`, run these commands in the same directory as the file:

### Create and run the model

```bash
# Create a Modelfile that points to the downloaded GGUF
echo "FROM ./model.gguf" > Modelfile

# Register the model with Ollama
ollama create my-finetuned-model -f Modelfile

# Run it interactively
ollama run my-finetuned-model
```

### Update your RAG chatbot to use the fine-tuned model

Edit `rag-chatbot-app/config.yaml` and update the generator model name:

```yaml
models:
  generator: "my-finetuned-model"
```

Then restart the chatbot — it will automatically use your fine-tuned model for all responses.